# DEPURACION DEFUNCIONES 2008 A 2011

## 1. IMPORTAR LIBRERIAS

In [6]:
### IMPORTAR LIBRERIAS ###
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## 2. CREAR FUNCIONES

### 2.1. FUNCION LLAMADO DE DATOS

In [8]:
def leer_archivos_defunciones(ruta_carpeta, año_inicio, año_fin, patron="Defun*"):
    """Lee y unifica archivos de defunciones en un rango de años
    
    Detecta automáticamente el tipo de archivo y separador
    """
    ruta = Path(ruta_carpeta)
    
    # Buscar archivos con extensiones comunes
    extensiones = ['.txt', '.csv', '.tsv']
    archivos = []
    
    for ext in extensiones:
        archivos.extend([arch for arch in ruta.glob(f"{patron}{ext}") 
                        if año_inicio <= int(arch.stem[-4:]) <= año_fin])
    
    dfs = []
    encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']
    
    for archivo in tqdm(archivos, desc="Leyendo archivos"):
        df = None
        extension = archivo.suffix.lower()
        
        # Determinar separador
        separadores = {
            '.txt': '\t',
            '.tsv': '\t',
            '.csv': ','
        }
        separador = separadores.get(extension, ',')
        
        for encoding in encodings:
            try:
                df = pd.read_csv(archivo, sep=separador, encoding=encoding, 
                               low_memory=False, dtype=str)
                df.columns = df.columns.str.upper()
                dfs.append(df)
                print(f"   ✓ {archivo.name} leído ({encoding})")
                break
            except UnicodeDecodeError:
                continue
            except Exception as e:
                if encoding == encodings[-1]:  # Solo mostrar en último intento
                    print(f"   ✗ Error en {archivo.name}: {e}")
                continue
        
        if df is None:
            print(f"   ✗ No se pudo leer {archivo.name}")
    
    if dfs:
        df_final = pd.concat(dfs, ignore_index=True)
        print(f"\n   📊 Total de archivos procesados: {len(dfs)}")
        return df_final
    else:
        print("\n   ⚠ No se encontraron archivos para procesar")
        return pd.DataFrame()

### 2.2. FUNCION PARA CREAR MAPEO DE DESCRIPCIONES

In [ ]:
def crear_mapeo_descripciones():
    """Crea diccionarios de mapeo para las descripciones"""

    # Generar rangos para PERMAN_MUN (01-98 años)
    permanencia_municipio = {
        'X1': 'MENOS DE 1 DÍA',
        'X2': 'DE 1 A 6 DÍAS',
        'X3': 'DE 7 A 29 DÍAS',
        'X4': 'DE 1 A 5 MESES',
        'X5': 'DE 6 A 11 MESES',
        '00': 'TIEMPO DESCONOCIDO',
        '99': 'DE 99 Y MÁS AÑOS'
    }

    # Agregar dinámicamente 01-98
    for i in range(1, 99):
        clave = f"{i:02d}"
        años = "AÑO" if i == 1 else "AÑOS"
        permanencia_municipio[clave] = f"{i} {años}"

    # Número de hijos vivos
    n_hijosv = {}
    for i in range(0, 21):
        texto = "HIJO NACIDO VIVO" if i == 1 else "HIJOS NACIDOS VIVOS"

        # Clave padded (01, 02, ..., 20)
        clave_padded = f"{i:02d}"
        # Clave sin padding (1, 2, ... 20)
        clave_simple = str(i)

        valor = f"{i} {texto}"

        n_hijosv[clave_padded] = valor   # "01"
        n_hijosv[clave_simple] = valor   # "1"

    # Caso SIN INFORMACIÓN
    n_hijosv['99'] = 'SIN INFORMACIÓN'

    # Número de hijos muertos
    n_hijosm = {}
    for i in range(1, 16):
        texto = "HIJO NACIDO MUERTO" if i == 1 else "HIJOS NACIDOS MUERTOS"

        # Clave padded (01, 02, ..., 20)
        clave_padded = f"{i:02d}"
        # Clave sin padding (1, 2, ... 20)
        clave_simple = str(i)

        valor = f"{i} {texto}"

        n_hijosm[clave_padded] = valor   # "01"
        n_hijosm[clave_simple] = valor   # "1"

    # Caso SIN INFORMACIÓN
    n_hijosm['99'] = 'SIN INFORMACIÓN'

    return {
        'A_DEFUN': {
            '1': 'CABECERA MUNICIPAL',
            '2': 'CENTRO POBLADO',
            '3': 'RURAL DISPERSO',
            '9': 'SIN INFORMACIÓN'
        },
        'SEXO': {
            '1': 'MASCULINO',
            '2': 'FEMENINO',
            '3': 'INDETERMINADO'
        },
        'EST_CIVIL': {
            '1': 'NO ESTABA CASADO(A) Y LLEVABA DOS O MAS AÑOS VIVIENDO CON SU PAREJA',
            '2': 'NO ESTABA CASADO(A) Y LLEVABA MENOS DE DOS AÑOS VIVIENDO CON SU PAREJA',
            '3': 'ESTABA SEPARADO(A), DIVORCIADO(A)',
            '4': 'ESTABA VIUDO(A)',
            '5': 'ESTABA SOLTERO(A)',
            '6': 'ESTABA CASADO(A)',
            '9': 'SIN INFORMACIÓN'
        },
        'SIT_DEFUN': {
            '1': 'HOSPITAL/CLÍNICA',
            '2': 'CENTRO/PUESTO DE SALUD',
            '3': 'CASA/DOMICILIO',
            '4': 'LUGAR DE TRABAJO',
            '5': 'VÍA PÚBLICA',
            '6': 'OTRO',
            '9': 'SIN INFORMACIÓN'
        },
        'GRU_ED1': {
            '00': 'MENOR DE UNA HORA',
            '01': 'MENOR DE UN DÍA', '02': 'DE 1 A 6 DÍAS',
            '03': 'DE 7 A 27 DÍAS', '04': 'DE 28 A 29 DÍAS',
            '05': 'DE 1 A 5 MESES',
            '06': 'DE 6 A 11 MESES', '07': 'DE UN AÑO',
            '08': 'DE 2 A 4 AÑOS', '09': 'DE 5 A 9 AÑOS',
            '10': 'DE 10 A 14 AÑOS', '11': 'DE 15 A 19 AÑOS',
            '12': 'DE 20 A 24 AÑOS', '13': 'DE 25 A 29 AÑOS',
            '14': 'DE 30 A 34 AÑOS', '15': 'DE 35 A 39 AÑOS',
            '16': 'DE 40 A 44 AÑOS', '17': 'DE 45 A 49 AÑOS',
            '18': 'DE 50 A 54 AÑOS', '19': 'DE 55 A 59 AÑOS',
            '20': 'DE 60 A 64 AÑOS', '21': 'DE 65 A 69 AÑOS',
            '22': 'DE 70 A 74 AÑOS', '23': 'DE 75 A 79 AÑOS',
            '24': 'DE 80 A 84 AÑOS', '25': 'DE 85 A 89 AÑOS',
            '26': 'DE 90 A 94 AÑOS', '27': 'DE 95 A 99 AÑOS',
            '28': 'DE 100 AÑOS Y MÁS', '29': 'EDAD DESCONOCIDA'
        },
        'GRU_ED2': {
            '01': 'MENOR DE UN AÑO', '02': 'DE 1 A 4 AÑOS',
            '03': 'DE 5 A 14 AÑOS', '04': 'DE 15 A 44 AÑOS',
            '05': 'DE 45 A 64 AÑOS', '06': 'DE 65 Y MÁS AÑOS',
            '07': 'EDAD DESCONOCIDA'
        },
        'CONS_EXP': {
            '1': 'MÉDICO TRATANTE',
            '2': 'MÉDICO NO TRATANTE',
            '3': 'MÉDICO LEGISTA',
            '4': 'PERSONAL DE SALUD AUTORIZADO',
            '5': 'PERSONAL DE REGISTRO CIVIL',
            '9': 'SIN INFORMACIÓN'
        },
        'AREA_RES': {
            '1': 'CABECERA',
            '2': 'CENTRO POBLADO',
            '3': 'RURAL DISPERSO',
            '9': 'SIN INFORMACIÓN'
        },
        'PMAN_MUER': {
            '1': 'NATURAL',
            '2': 'VIOLENTA',
            '3': 'EN ESTUDIO'
        },
        'TIPO_DEFUN': {
            '1': 'DEFUNCIÓN FETAL',
            '2': 'DEFUNCIÓN NO FETAL'
        },
        'NIVEL_EDU': {
            '1': 'PREESCOLAR',
            '2': 'PRIMARIA COMPLETA',
            '3': 'PRIMARIA INCOMPLETA',
            '4': 'SECUNDARIA COMPLETA',
            '5': 'SECUNDARIA INCOMPLETA',
            '6': 'UNIVERSITARIA COMPLETA',
            '7': 'UNIVERSITARIA INCOMPLETA',
            '8': 'NINGUNO',
            '9': 'SIN INFORMACIÓN'
        },
        'CODPRES': {
            '076': 'BRASIL',
            '100': 'BULGARIA',
            '170': 'COLOMBIA',
            '188': 'COSTA RICA',
            '218': 'ECUADOR',
            '250': 'FRANCIA',
            '380': 'ITALIA',
            '533': 'ARUBA',
            '591': 'PANAMA',
            '604': 'PERU',
            '724': 'ESPAÑA',
            '840': 'ESTADOS UNIDOS DE AMÉRICA',
            '862': 'VENEZUELA',
            '998': 'RESTO DE PAISES',
            '999': 'SIN INFORMACIÓN'
        },
        'CODPTORE': {
            '01': 'SIN INFORMACIÓN DE DEPARTAMENTO',
            '75': 'CON RESIDENCIA EN EL EXTRANJERO',
            '05': 'ANTIOQUIA',
            '08': 'ATLÁNTICO',
            '11': 'BOGOTÁ D.C.',
            '13': 'BOLÍVAR',
            '15': 'BOYACÁ',
            '17': 'CALDAS',
            '18': 'CAQUETÁ',
            '19': 'CAUCA',
            '20': 'CESAR',
            '23': 'CÓRDOBA',
            '25': 'CUNDINAMARCA',
            '27': 'CHOCÓ',
            '41': 'HUILA',
            '44': 'LA GUAJIRA',
            '47': 'MAGDALENA',
            '50': 'META',
            '52': 'NARIÑO',
            '54': 'NORTE DE SANTANDER',
            '63': 'QUINDÍO',
            '66': 'RISARALDA',
            '68': 'SANTANDER',
            '70': 'SUCRE',
            '73': 'TOLIMA',
            '76': 'VALLE DEL CAUCA',
            '81': 'ARAUCA',
            '85': 'CASANARE',
            '86': 'PUTUMAYO',
            '88': 'ARCHIPIÉLAGO DE SAN ANDRÉS, PROVIDENCIA Y SANTA CATALINA',
            '91': 'AMAZONAS',
            '94': 'GUAINÍA',
            '95': 'GUAVIARE',
            '97': 'VAUPÉS',
            '99': 'VICHADA'
        },
        'SEG_SOCIAL': {
            '1': 'CONTRIBUTIVO',
            '2': 'SUBSIDIADO',
            '3': 'VINCULADO',
            '4': 'IGNORADO',
            '9': 'SIN INFORMACIÓN'
        },
        'MU_PARTO': {
            '1': 'ANTES',
            '2': 'DURANTE',
            '3': 'DESPUÉS',
            '4': 'IGNORADO',
            '9': 'SIN INFORMACIÓN'
        },
        'T_PARTO': {
            '1': 'ESPONTÁNEO',
            '2': 'CESÁREA',
            '3': 'INSTRUMENTADO',
            '4': 'IGNORADO',
            '9': 'SIN INFORMACIÓN'
        },
        'TIPO_EMB': {
            '1': 'SIMPLE',
            '2': 'MÚLTIPLE',
            '9': 'SIN INFORMACIÓN'
        },
        'T_GES': {
            '1': 'MENOS DE 20',
            '2': 'DE 20 A 27',
            '3': 'DE 28 Y MÁS',
            '4': 'IGNORADO',
            '9': 'SIN INFORMACIÓN'
        },
        'PESO_NAC': {
            '1': 'MENOS DE 1000 GRAMOS',
            '2': 'DE 1000 A 1499 GRAMOS',
            '3': 'DE 1500 A 1999 GRAMOS',
            '4': 'DE 2000 A 2499 GRAMOS',
            '5': 'DE 2500 A 2999 GRAMOS',
            '6': 'DE 3000 A 3499 GRAMOS',
            '7': 'DE 3500 A 3999 GRAMOS',
            '8': 'DE 4000 Y MÁS GRAMOS',
            '9': 'SIN INFORMACIÓN'
        },
        'EDAD_MADRE': {
            '1': 'DE 10 A 14 AÑOS',
            '2': 'DE 15 A 19 AÑOS',
            '3': 'DE 20 A 24 AÑOS',
            '4': 'DE 25 A 29 AÑOS',
            '5': 'DE 30 A 34 AÑOS',
            '6': 'DE 35 A 39 AÑOS',
            '7': 'DE 40 A 44 AÑOS',
            '8': 'DE 45 A 49 AÑOS',
            '9': 'DE 50 A 54 AÑOS',
            '99': 'SIN INFORMACIÓN'
        },
        'EST_CIVM': {
            '1': 'SOLTERA',
            '2': 'CASADA',
            '3': 'VIUDA',
            '4': 'EN UNIÓN LIBRE',
            '5': 'SEPARADA O DIVORCIADA',
            '9': 'SIN INFORMACIÓN'
        },
        'NIV_EDUM': {
            '1': 'PREESCOLAR',
            '2': 'PRIMARIA COMPLETA',
            '3': 'PRIMARIA INCOMPLETA',
            '4': 'SECUNDARIA COMPLETA',
            '5': 'SECUNDARIA INCOMPLETA',
            '6': 'UNIVERSITARIA COMPLETA',
            '7': 'UNIVERSITARIA INCOMPLETA',
            '8': 'NINGUNO',
            '9': 'SIN INFORMACIÓN'
        },
        'EMB_FAL': {
            '1': 'SI',
            '2': 'NO',
            '9': 'SIN INFORMACIÓN'
        },
        'EMB_SEM': {
            '1': 'SI',
            '2': 'NO',
            '9': 'SIN INFORMACIÓN'
        },
        'EMB_MES': {
            '1': 'SI',
            '2': 'NO',
            '9': 'SIN INFORMACIÓN'
        },
        'MAN_MUER': {
            '1': 'SUICIDIO',
            '2': 'HOMICIDIO',
            '3': 'ACCIDENTE DE TRÁNSITO',
            '4': 'OTRO ACCIDENTE',
            '5': 'EN ESTUDIO',
            '9': 'SIN INFORMACIÓN'
        },
        'CODOCUR': {
            '01': 'SIN INFORMACIÓN DE DEPARTAMENTO',
            '75': 'CON RESIDENCIA EN EL EXTRANJERO',
            '05': 'ANTIOQUIA',
            '08': 'ATLÁNTICO',
            '11': 'BOGOTÁ D.C.',
            '13': 'BOLÍVAR',
            '15': 'BOYACÁ',
            '17': 'CALDAS',
            '18': 'CAQUETÁ',
            '19': 'CAUCA',
            '20': 'CESAR',
            '23': 'CÓRDOBA',
            '25': 'CUNDINAMARCA',
            '27': 'CHOCÓ',
            '41': 'HUILA',
            '44': 'LA GUAJIRA',
            '47': 'MAGDALENA',
            '50': 'META',
            '52': 'NARIÑO',
            '54': 'NORTE DE SANTANDER',
            '63': 'QUINDÍO',
            '66': 'RISARALDA',
            '68': 'SANTANDER',
            '70': 'SUCRE',
            '73': 'TOLIMA',
            '76': 'VALLE DEL CAUCA',
            '81': 'ARAUCA',
            '85': 'CASANARE',
            '86': 'PUTUMAYO',
            '88': 'ARCHIPIÉLAGO DE SAN ANDRÉS, PROVIDENCIA Y SANTA CATALINA',
            '91': 'AMAZONAS',
            '94': 'GUAINÍA',
            '95': 'GUAVIARE',
            '97': 'VAUPÉS',
            '99': 'VICHADA'
        },
        'C_MUERTE': {
            '1': 'NECROPSIA',
            '2': 'HISTORIA CLÍNICA',
            '3': 'PRUEBAS DE LABORATORIO',
            '4': 'INTERROGATORIO A FAMILIARES O TESTIGOS',
            '9': 'SIN INFORMACIÓN',
        },
        'ASIS_MED': {
            '1': 'SÍ',
            '2': 'NO',
            '3': 'IGNORADO',
            '9': 'SIN INFORMACIÓN',
        },
        'N_HIJOSV': n_hijosv,
        'N_HIJOSM': n_hijosm,
        'TIEM_PER': permanencia_municipio
    }

## 3. EJECUTAR PROCESO PRINCIPAL

### 3.1. DESPLIEGUE PROCESO PRINCIPAL

In [4]:
# =============================================================================
# 3. PROCESO PRINCIPAL
# =============================================================================

print("=" * 60)
print("ANÁLISIS EXPLORATORIO DE DATOS - DEFUNCIONES 2008-2011")
print("=" * 60)

ANÁLISIS EXPLORATORIO DE DATOS - DEFUNCIONES 2008-2011


### 3.2. LECTURA DE ARCHIVOS DE DEFUNCIONES

In [9]:
####################################
# 3.1. Leer archivos de defunciones#
####################################
print("\n1. LEYENDO ARCHIVOS DE DEFUNCIONES...")
df_defun = leer_archivos_defunciones("data/raw/Muertes", 2008, 2011)
print(f"   Total registros: {len(df_defun):,}")
print(f"   Columnas: {list(df_defun.columns)}")
print(f"   Número de Columnas: {len(list(df_defun.columns))}")
pd.set_option('display.max_columns', None) # Mostrar todas las columnas
display(df_defun.head())


1. LEYENDO ARCHIVOS DE DEFUNCIONES...


Leyendo archivos:  25%|██▌       | 1/4 [00:02<00:06,  2.33s/it]

   ✓ Defun2008.csv leído (latin-1)


Leyendo archivos:  50%|█████     | 2/4 [00:04<00:04,  2.30s/it]

   ✓ Defun2009.csv leído (latin-1)


...

   ✓ Defun2010.csv leído (latin-1)


Leyendo archivos: 100%|██████████| 4/4 [00:09<00:00,  2.34s/it]

   ✓ Defun2011.csv leído (latin-1)

   📊 Total de archivos procesados: 4


   Total registros: 790,223
   Columnas: ['COD_DPTO', 'COD_MUNIC', 'A_DEFUN', 'SIT_DEFUN', 'OTRSITIODE', 'TIPO_DEFUN', 'ANO', 'MES', 'HORA', 'MINUTOS', 'SEXO', 'EST_CIVIL', 'GRU_ED1', 'GRU_ED2', 'NIVEL_EDU', 'ULTCURFAL', 'MUERTEPORO', 'SIMUERTEPO', 'OCUPACION', 'IDPERTET', 'CODPRES', 'CODPTORE', 'CODMUNRE', 'AREA_RES', 'SEG_SOCIAL', 'IDADMISALUD', 'PMAN_MUER', 'CONS_EXP', 'MU_PARTO', 'T_PARTO', 'TIPO_EMB', 'T_GES', 'T_GES_AGRU_CIE', 'PESO_NAC', 'EDAD_MADRE', 'N_HIJOSV', 'N_HIJOSM', 'EST_CIVM', 'NIV_EDUM', 'ULTCURMAD', 'EMB_FAL', 'EMB_SEM', 'EMB_MES', 'MAN_MUER', 'CODOCUR', 'CODMUNOC', 'C_MUERTE', 'C_MUERTEB', 'C_MUERTEC', 'C_MUERTED', 'C_MUERTEE', 'ASIS_MED', 'C_DIR1', 'C_DIR12', 'C_ANT1', 'C_ANT12', 'C_ANT2', 'C_ANT22', 'C_ANT3', 'C_ANT32', 'C_PAT1', 'C_PAT2', 'C_MCM1', 'C_BAS1', 'CAUSA_666', 'CAUSA_667', 'IDPROFCER', 'CAU_HOMOL']
   Número de Columnas: 68


,COD_DPTO,COD_MUNIC,A_DEFUN,SIT_DEFUN,OTRSITIODE,TIPO_DEFUN,ANO,MES,HORA,MINUTOS,SEXO,EST_CIVIL,GRU_ED1,GRU_ED2,NIVEL_EDU,ULTCURFAL,MUERTEPORO,SIMUERTEPO,OCUPACION,IDPERTET,CODPRES,CODPTORE,CODMUNRE,AREA_RES,SEG_SOCIAL,IDADMISALUD,PMAN_MUER,CONS_EXP,MU_PARTO,T_PARTO,TIPO_EMB,T_GES,T_GES_AGRU_CIE,PESO_NAC,EDAD_MADRE,N_HIJOSV,N_HIJOSM,EST_CIVM,NIV_EDUM,ULTCURMAD,EMB_FAL,EMB_SEM,EMB_MES,MAN_MUER,CODOCUR,CODMUNOC,C_MUERTE,C_MUERTEB,C_MUERTEC,C_MUERTED,C_MUERTEE,ASIS_MED,C_DIR1,C_DIR12,C_ANT1,C_ANT12,C_ANT2,C_ANT22,C_ANT3,C_ANT32,C_PAT1,C_PAT2,C_MCM1,C_BAS1,CAUSA_666,CAUSA_667,IDPROFCER,CAU_HOMOL
0,68,001,1,1,NaN,2,2008,02,04,00,1,4,25,06,99,99,9,9,NaN,9,170,68,001,1,9,9,2,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,NaN,NaN,NaN,4,68,001,1,NaN,NaN,NaN,NaN,1,T794,NaN,K922,NaN,K290,NaN,S720,NaN,R54X,NaN,NaN,W189,503,503,1,093
1,68,001,1,1,NaN,2,2008,02,08,15,1,9,22,06,99,99,9,9,NaN,9,170,68,001,1,2,2,1,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,1,I249,NaN,I10X,NaN,I420,NaN,NaN,NaN,NaN,NaN,NaN,I249,303,303,1,051
2,68,001,1,1,NaN,2,2008,02,07,50,1,6,19,05,4,99,9,9,NaN,9,170,68,001,1,3,5,1,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,9,R570,NaN,I119,NaN,NaN,NaN,NaN,NaN,F171,NaN,NaN,I119,302,302,1,050
3,68,001,1,1,NaN,2,2008,02,01,45,1,9,22,06,99,99,9,9,NaN,9,170,68,001,1,2,2,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,1,R578,NaN,E878,NaN,NaN,NaN,NaN,NaN,N189,NaN,NaN,N189,612,610,1,074
4,68,001,1,1,NaN,2,2008,02,02,00,1,9,18,05,99,99,9,9,NaN,9,170,68,001,1,2,2,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,1,A419,NaN,K659,NaN,K275,NaN,NaN,NaN,NaN,NaN,NaN,K275,611,609,1,063
